# Phase 3: Capstone Mentor AI Fine-Tuning
This notebook fine-tunes the Qwen 2.5 7B Instruct model using your `train.jsonl` dataset.

### Step 1: Install Dependencies
We use Unsloth because it makes training twice as fast and uses significantly less memory.

In [ ]:
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes


### Step 2: Load the Base Model from Hugging Face
This downloads the 15GB model directly to Kaggle's servers in about 60 seconds.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Good for our prompt length

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit", # 4-bit compressed version
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Configure LoRA Adapters (which layers to train)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


### Step 3: Load and Format the Dataset
Upload your `train.jsonl` file to Kaggle, then run this cell.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Ensure you uploaded train.jsonl to the Kaggle working directory!
dataset = load_dataset("json", data_files="/kaggle/input/datasets/shyamsaranp/fine-tune-data/train.jsonl", split="train")

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Qwen uses ChatML format
)

def format_chat_template(examples):
    texts = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in examples["messages"]]
    return {"text": texts}

dataset = dataset.map(format_chat_template, batched=True)
print(f"Loaded and formatted {len(dataset)} samples successfully!")


### Step 4: Start Fine-Tuning
This will take roughly **5 to 10 minutes** for 100 samples. Watch the progress bar below!

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Adjust depending on dataset size. 60 is good for testing.
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()


### Step 5: Test It Live
Test the newly fine-tuned model!

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# Quick test message
test_messages = [
    {"role": "system", "content": "You are Capstone Mentor AI. Answer strictly based on the context..."},
    {"role": "user", "content": "I am stuck on module 3, what should I use?"}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 256, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])


### Step 6: Export the LoRA Adapters
Save the weights so you can download them from Kaggle.

In [ ]:
model.save_pretrained("lora_capstone_mentor")
tokenizer.save_pretrained("lora_capstone_mentor")
print("Saved! You can now download the lora_capstone_mentor folder.")
